# Mask blob-merge fix: erosion vs. watershed, and why watershed won

Landmarks close together in `mean_coords` (1/2 is the closest pair of all 19,
see `jobs/online/README.md`) can have their circular mask targets touch or
overlap in the predicted mask -- especially at the larger mask radii configs
6/7 use (radius 4, up from 2), chosen for rotation robustness and DeepWings
generalization. `cv2.findContours` then sees the two touching blobs as one
connected component and returns a single centroid roughly *between* the two
true landmark positions, instead of two separate ones -- visually confirmed
on `PL-0012-000270-R.dw.png` (a real wing outside the labeled dataset): a
clear "figure-8" shaped blob, area ~2x the median single-landmark blob.

Two candidate fixes for `mask_to_coords`, compared here across the whole
internal test set (not just the one example that surfaced this):
- **Erosion** (3x3 elliptical kernel, 2 iterations) before finding contours:
  enough to break the thin neck connecting two touching circles before
  eating away the circles themselves, splitting them back into two contours.
  Applied unconditionally to *every* blob, not just merged ones.
- **Watershed**: distance-transform + local-maxima peak detection, splitting
  only contours flagged as anomalously large relative to the image's own
  median blob area. Normal, isolated blobs never enter this path at all.

Measured below: how many wings actually show this merge signature; how many
of those each fix resolves; whether accuracy (not just point count)
improves; whether erosion ever *regresses* an already-correct detection
(processing every blob unconditionally risks eroding away an already-small,
already-separate blob entirely); and whether watershed's extra complexity
costs meaningfully more time. **Result: watershed matches erosion's fix rate
and per-image cost with no regressions, so `wings/visualizing/image_preprocess.py`'s
`mask_to_coords` ships the watershed version.**

## Setup

In [1]:
import cv2
import numpy as np
import torch
from torchvision import tv_tensors

from wings.config import PROCESSED_DATA_DIR, MODELS_DIR, COUNTRIES
from wings.dataset import WingsRawDataset
from wings.transforms import build_eval_transform
from wings.metrics import gpa_ordered_metrics_single
from wings.modeling.unet import UNet
from wings.modeling.litnet import LitNet
from wings.modeling.loss import BCEDiceLoss

torch.manual_seed(0)

CHECKPOINT_PATH = MODELS_DIR / "online" / "last-7.ckpt"  # config 7's trained checkpoint
OUTPUT_SIZE = 400
AREA_RATIO = 1.5  # a contour bigger than this x the image's own median blob area is flagged as a likely merge

mean_coords = torch.load(
    PROCESSED_DATA_DIR / "mask_datasets" / "rectangle" / "mean_shape.pth", weights_only=False
)

model = UNet(in_channels=1, out_channels=1, kernel_size=5, sigmoid=False)
lit_net = LitNet.load_from_checkpoint(
    CHECKPOINT_PATH, model=model, criterion=BCEDiceLoss(), num_epochs=1,
    mean_coords=mean_coords, strict=False,
)
lit_net.eval()
device = lit_net.device
print(f"Loaded checkpoint: {CHECKPOINT_PATH.name}")

raw = WingsRawDataset(COUNTRIES, PROCESSED_DATA_DIR / "cropped")
_, _, test_idx = raw.split(val_percentage=0.2, test_percentage=0.1, seed=42)
eval_transform = build_eval_transform(OUTPUT_SIZE)
print(f"Test samples: {len(test_idx)}")

2026-09-27 21:47:50.049 | INFO     | wings.config:<module>:44 - PROJ_ROOT path is: C:\Users\X\projects\bees


2026-09-27 21:47:50.078 | INFO     | wings.config:<module>:68 - torch.cuda.get_device_name()='NVIDIA RTX A3000 12GB Laptop GPU'


W0927 21:47:51.959000 10664 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loaded checkpoint: last-7.ckpt


  0%|          | 0/21722 [00:00<?, ?it/s]

 26%|██▌       | 5609/21722 [00:00<00:00, 55634.56it/s]

 76%|███████▋  | 16612/21722 [00:00<00:00, 87283.72it/s]

100%|██████████| 21722/21722 [00:00<00:00, 58757.76it/s]

Test samples: 2172


## Helper functions

Three decoders sharing the same raw predicted mask (the model only runs
once per image; only the decode step differs). All three are defined
locally in this notebook rather than imported from `wings/`, so this
comparison stays a fixed historical snapshot regardless of which one
`mask_to_coords` actually ships with later (as of this notebook, it ships
`decode_watershed`'s logic -- see `wings/visualizing/image_preprocess.py`):
- `decode_baseline` -- the original logic (threshold + contours + moments,
  no fix at all) -- measures how often the merge signature occurred before
  any fix.
- `decode_eroded` -- erosion (3x3 elliptical kernel, 2 iterations) before
  finding contours -- the first fix tried.
- `decode_watershed` -- distance-transform + local-maxima peak detection,
  splitting only contours flagged as anomalously large -- the one actually
  adopted, per this notebook's results below.

In [2]:
def predict_mask(image_tensor):
    """image_tensor: (1, H, W), already normalized. Returns the rounded
    (0/1) predicted mask as a numpy array, same as compute_statistics/
    wings.app.images use."""
    with torch.no_grad():
        output = lit_net.model(image_tensor.unsqueeze(0).to(device))
    probs = torch.sigmoid(output)
    return torch.round(probs).squeeze().detach().cpu().numpy()


def _binary_and_contours(mask):
    binary = (mask * 255).astype(np.uint8)
    _, binary = cv2.threshold(binary, 200, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return binary, contours


def decode_baseline(mask):
    """Pre-fix mask_to_coords: no erosion, one centroid per contour."""
    _, contours = _binary_and_contours(mask)
    img_y_size = mask.shape[0]
    coords = []
    for cnt in contours:
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
            coords.append((cx, img_y_size - cy - 1))
    return coords


def decode_eroded(mask, ksize=3, iterations=2):
    """Erosion before finding contours -- the first fix tried (see git
    history of wings/visualizing/image_preprocess.py). Applied
    unconditionally to every blob, not just merged ones."""
    binary = (mask * 255).astype(np.uint8)
    _, binary = cv2.threshold(binary, 200, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    binary = cv2.erode(binary, kernel, iterations=iterations)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_y_size = mask.shape[0]
    coords = []
    for cnt in contours:
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
            coords.append((cx, img_y_size - cy - 1))
    return coords


def contour_anomaly(mask):
    """True if any contour's area exceeds AREA_RATIO x the image's own
    median contour area -- the blob-merge signature. Needs >=3 contours for
    a stable median (skip the check on very sparse/broken predictions)."""
    _, contours = _binary_and_contours(mask)
    if len(contours) < 3:
        return False, len(contours)
    areas = np.array([cv2.contourArea(c) for c in contours])
    return bool((areas > AREA_RATIO * np.median(areas)).any()), len(contours)


def split_blob_watershed(roi_binary):
    """roi_binary: (h, w) uint8, 0/255, a single (possibly merged) blob.
    Returns a list of (x, y) centroids, or None if fewer than 2 peaks are found."""
    dist = cv2.distanceTransform(roi_binary, cv2.DIST_L2, 5)
    dist_max = dist.max()
    if dist_max == 0:
        return None
    kernel = np.ones((5, 5), np.uint8)
    dilated = cv2.dilate(dist, kernel)
    local_max = (dist == dilated) & (dist > 0.5 * dist_max)
    n_labels, markers = cv2.connectedComponents(local_max.astype(np.uint8))
    if n_labels <= 2:
        return None
    markers = markers + 1
    markers[roi_binary == 0] = 0
    roi_bgr = cv2.cvtColor(roi_binary, cv2.COLOR_GRAY2BGR)
    cv2.watershed(roi_bgr, markers)
    centroids = []
    for label in range(2, n_labels + 1):
        ys, xs = np.where(markers == label)
        if len(xs) == 0:
            continue
        centroids.append((xs.mean(), ys.mean()))
    return centroids


def decode_watershed(mask, pad=5):
    """The decoder this notebook's results led to shipping (see
    wings/visualizing/image_preprocess.py's mask_to_coords): baseline
    centroids, except contours flagged as anomalously large are split via
    distance-transform + watershed into one centroid per detected peak.
    Reimplemented locally here rather than imported, so this notebook stays
    a fixed comparison snapshot. Normal contours are completely
    untouched (unlike erosion, which processes every blob)."""
    binary, contours = _binary_and_contours(mask)
    if len(contours) < 3:
        return decode_baseline(mask)
    areas = np.array([cv2.contourArea(c) for c in contours])
    median_area = np.median(areas)
    img_y_size = mask.shape[0]

    coords = []
    for cnt, area in zip(contours, areas):
        if area > AREA_RATIO * median_area:
            x, y, w, h = cv2.boundingRect(cnt)
            y0, y1_ = max(0, y - pad), min(binary.shape[0], y + h + pad)
            x0, x1_ = max(0, x - pad), min(binary.shape[1], x + w + pad)
            roi = binary[y0:y1_, x0:x1_].copy()
            roi_mask = np.zeros_like(roi)
            cv2.drawContours(roi_mask, [cnt - [x0, y0]], -1, 255, thickness=cv2.FILLED)
            roi = cv2.bitwise_and(roi, roi_mask)
            split = split_blob_watershed(roi)
            if split:
                for px, py in split:
                    coords.append((px + x0, img_y_size - (py + y0) - 1))
                continue
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
            coords.append((cx, img_y_size - cy - 1))
    return coords


def run_sample(base_index):
    """Deterministic eval-transform sample: transformed image + ground
    truth in canonical landmark order (same approach as notebook 25)."""
    image, orig_label, orig_size = test_idx[base_index]
    orig_width, orig_height = orig_size
    x = orig_label[::2].clone()
    y_top = orig_height - orig_label[1::2].clone() - 1
    kp = torch.stack([x, y_top], dim=-1)
    image_tv = tv_tensors.Image(image)
    kp_tv = tv_tensors.KeyPoints(kp, canvas_size=(orig_height, orig_width))

    t_image, t_kp = eval_transform(image_tv, kp_tv)
    t_image = t_image.as_subclass(torch.Tensor)
    t_kp = t_kp.as_subclass(torch.Tensor)

    final_size = t_image.shape[-1]
    true_x = t_kp[:, 0]
    true_y_bottom = final_size - t_kp[:, 1] - 1
    true_ordered = torch.stack([true_x, true_y_bottom], dim=-1)
    return t_image, true_ordered

## Main evaluation loop

Runs the model once per test image; decodes the same predicted mask three
ways and records point counts, plus (for wings flagged with the merge
signature) GPA-ordered pixel error and positional precision under each
decoder, against the same ground truth.

In [3]:
from wings.gpa import FULL_ROTATION_MULTISTART_ANGLES

results = []
cached_masks = []  # reused by the timing cell below, so predict() only ever runs once per image
for idx in range(len(test_idx)):
    t_image, true_ordered = run_sample(idx)
    mask = predict_mask(t_image)
    cached_masks.append(mask.astype(np.uint8))

    anomaly, n_baseline = contour_anomaly(mask)
    base_coords = decode_baseline(mask)
    eroded_coords = decode_eroded(mask)
    watershed_coords = decode_watershed(mask)

    row = {
        "idx": idx,
        "anomaly": anomaly,
        "n_baseline": len(base_coords),
        "n_eroded": len(eroded_coords),
        "n_watershed": len(watershed_coords),
    }

    if anomaly:
        for name, coords in [("baseline", base_coords), ("eroded", eroded_coords), ("watershed", watershed_coords)]:
            d, precision = gpa_ordered_metrics_single(
                coords, true_ordered, mean_coords, allow_reflection=True,
                multistart_angles=FULL_ROTATION_MULTISTART_ANGLES,
            )
            row[f"{name}_mean_px"] = d.mean().item() if d is not None else float("nan")
            row[f"{name}_precision"] = precision

    results.append(row)

print(f"Evaluated {len(results)} test wings.")

Evaluated 2172 test wings.


## Results

In [4]:
import pandas as pd

df = pd.DataFrame(results)

n_total = len(df)
n_anomaly = df["anomaly"].sum()
n_missing_no_anomaly = ((df["n_baseline"] < 19) & (~df["anomaly"])).sum()

print(f"Total wings evaluated:                                    {n_total}")
print(f"Wings with a blob-merge signature (oversized contour):     {n_anomaly}  ({100*n_anomaly/n_total:.2f}%)")
print(f"Wings with <19 points but NO oversized blob")
print(f"  (genuine missed detection, no blob to split -- a decode-level")
print(f"  fix can't help these, this is a segmentation-quality issue):  {n_missing_no_anomaly}")
print()

anomaly_df = df[df["anomaly"]]
resolved_eroded = (anomaly_df["n_eroded"] == 19).sum()
resolved_watershed = (anomaly_df["n_watershed"] == 19).sum()
print(f"Of the {n_anomaly} flagged wings:")
print(f"  resolved to exactly 19 points by erosion:    {resolved_eroded}  ({100*resolved_eroded/max(n_anomaly,1):.1f}%)")
print(f"  resolved to exactly 19 points by watershed:  {resolved_watershed}  ({100*resolved_watershed/max(n_anomaly,1):.1f}%)")
print()

# Regression check: did erosion ever REDUCE the point count on wings that
# had no merge signature to begin with (eroding away a small/faint blob)?
clean_df = df[~df["anomaly"]]
regressed = (clean_df["n_eroded"] < clean_df["n_baseline"]).sum()
print(f"Regression check -- clean wings (no anomaly) where erosion REDUCED the point count: {regressed} / {len(clean_df)}")
if regressed > 0:
    print("  (these wings, if any, are worth a closer look -- erosion may be too aggressive there)")
    print(clean_df[clean_df["n_eroded"] < clean_df["n_baseline"]][["idx", "n_baseline", "n_eroded", "n_watershed"]])
print()

print(f"{'metric':30s} {'baseline':>10s} {'eroded':>10s} {'watershed':>10s}")
for metric, label in [("mean_px", "GPA-ordered mean (px)"), ("precision", "positional precision")]:
    vals = []
    for name in ["baseline", "eroded", "watershed"]:
        col = f"{name}_{metric}"
        vals.append(anomaly_df[col].mean())
    print(f"{label:30s} {vals[0]:10.4f} {vals[1]:10.4f} {vals[2]:10.4f}")
print("(computed only over the flagged/anomaly subset -- the wings this fix targets)")

Total wings evaluated:                                    2172
Wings with a blob-merge signature (oversized contour):     103  (4.74%)
Wings with <19 points but NO oversized blob
  (genuine missed detection, no blob to split -- a decode-level
  fix can't help these, this is a segmentation-quality issue):  4

Of the 103 flagged wings:
  resolved to exactly 19 points by erosion:    98  (95.1%)
  resolved to exactly 19 points by watershed:  99  (96.1%)

Regression check -- clean wings (no anomaly) where erosion REDUCED the point count: 29 / 2069
  (these wings, if any, are worth a closer look -- erosion may be too aggressive there)
       idx  n_baseline  n_eroded  n_watershed
96      96          22        21           22
142    142          20        19           20
259    259          19        18           19
266    266          20        19           20
322    322          20        19           20
724    724          20        19           20
759    759          20        19         

## Timing: is watershed's extra complexity actually more expensive?

Decode-only timing (model inference already ran once above and isn't
repeated here -- `cached_masks` holds every predicted mask). Each decoder
runs several passes over all 2172 cached masks for a stable per-image
average.

In [5]:
import time

N_REPEATS = 5
timings = {}
for name, fn in [("baseline", decode_baseline), ("eroded", decode_eroded), ("watershed", decode_watershed)]:
    t0 = time.perf_counter()
    for _ in range(N_REPEATS):
        for m in cached_masks:
            fn(m)
    elapsed = time.perf_counter() - t0
    timings[name] = 1000 * elapsed / (N_REPEATS * len(cached_masks))
    print(f"{name:12s} {timings[name]:.4f} ms/image  (over {N_REPEATS} passes x {len(cached_masks)} images)")

print()
print(f"eroded vs baseline:    +{timings['eroded'] - timings['baseline']:.4f} ms/image")
print(f"watershed vs baseline: +{timings['watershed'] - timings['baseline']:.4f} ms/image")
print(f"watershed vs eroded:   {timings['watershed'] - timings['eroded']:+.4f} ms/image")
print()
print("For scale: model inference alone runs at roughly 27 ms/image on this GPU")
print("(notebook 26's full production pipeline, including handle_coordinates'")
print("combinatorial matching, measured ~228 ms/image) -- either decoder's cost")
print("is a small fraction of a single model forward pass, let alone the full pipeline.")

baseline     0.2428 ms/image  (over 5 passes x 2172 images)


eroded       0.2824 ms/image  (over 5 passes x 2172 images)


watershed    0.2867 ms/image  (over 5 passes x 2172 images)

eroded vs baseline:    +0.0395 ms/image
watershed vs baseline: +0.0438 ms/image
watershed vs eroded:   +0.0043 ms/image

For scale: model inference alone runs at roughly 27 ms/image on this GPU
(notebook 26's full production pipeline, including handle_coordinates'
combinatorial matching, measured ~228 ms/image) -- either decoder's cost
is a small fraction of a single model forward pass, let alone the full pipeline.


## Summary

- **How often**: the percentage of test wings above shows how common the
  blob-merge signature actually is on this checkpoint/mask radius -- rare
  enough to be missed in casual spot-checks (like the one that surfaced
  this on an out-of-dataset image), common enough to be worth fixing if
  it's more than a handful of wings.
- **Erosion vs. watershed resolution rate**: if erosion resolves nearly as
  many flagged wings as watershed, the simpler fix (already shipped in
  `mask_to_coords`) is justified. If watershed clearly resolves
  meaningfully more, that's the concrete case for switching to it despite
  the added complexity.
- **Regression check**: erosion processes every blob, not just merged ones
  -- any wing where it reduced the point count on an already-clean
  detection is a real cost specific to erosion that watershed's
  gated/anomaly-only approach doesn't have.
- **Accuracy, not just count**: resolving to 19 points doesn't automatically
  mean the two split points are individually accurate -- the GPA-ordered
  mean error and positional precision on the flagged subset show whether
  the fix actually helps positional accuracy, not just the raw count used
  to decide "clean 19-landmark detection" elsewhere in this project.
- **Timing**: gating the expensive part (distance transform + watershed) on
  only the anomalously-large contours -- rare, and processed on a small
  local crop, not the whole mask -- keeps watershed's average per-image cost
  statistically indistinguishable from erosion's, which runs unconditionally
  but on the whole mask at once. Speed is not a reason to prefer one over
  the other here; both are a small fraction of a single model forward pass.